# Q-Learning: Latihan Interaktif

**🎯 Tujuan Pembelajaran:**
1. Memahami konsep Q-Learning
2. Implementasi epsilon-greedy policy
3. Analisis learning curves
4. Eksperimen dengan hyperparameters
5. Evaluasi dan perbandingan policy

**📋 Prerequisites:**
- Dasar reinforcement learning
- Pemahaman tentang MDP (Markov Decision Process)
- Python dan NumPy

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('../..')

from implementation import (
    GridWorldSPG, QLearningAgent, RandomAgent,
    plot_learning_curve, compare_policies
)

# Untuk reproducibility
SEED = 42
np.random.seed(SEED)

print("✓ Libraries imported successfully!")

## Bagian 1: Memahami Environment

Grid World SPG adalah environment sederhana di mana:
- SPG (agent) harus mengunjungi outlet-outlet
- Ada hambatan yang harus dihindari
- Setiap langkah ada penalty kecil (-0.1)
- Mengunjungi outlet baru: reward +10
- Menabrak hambatan: penalty -10

In [ ]:
# Buat environment
env = GridWorldSPG(grid_size=5, n_outlets=3, n_obstacles=3, seed=SEED)

# Reset dan visualisasi
env.reset()
fig, ax = env.render()
plt.title('SPG Grid World - Initial State')
plt.show()

print(f"Grid Size: {env.grid_size}x{env.grid_size}")
print(f"Number of States: {env.grid_size ** 2}")
print(f"Number of Actions: {len(env.actions)}")
print(f"Actions: {env.action_names}")
print(f"\nOutlet Positions: {env.outlet_positions}")
print(f"Obstacle Positions: {env.obstacle_positions}")

### 🎯 Latihan 1.1: Eksplorasi Manual

Coba ambil beberapa langkah manual dan lihat reward yang didapat.

In [ ]:
# Reset environment
state = env.reset()
print(f"Start Position: {state}\n")

# Ambil beberapa langkah
# Actions: 0=up, 1=right, 2=down, 3=left
actions = [1, 1, 2, 2, 3]  # Contoh: kanan, kanan, bawah, bawah, kiri

total_reward = 0
for i, action in enumerate(actions):
    next_state, reward, done, info = env.step(action)
    total_reward += reward
    
    print(f"Step {i+1}:")
    print(f"  Action: {env.action_names[action]}")
    print(f"  Next State: {next_state}")
    print(f"  Reward: {reward:.2f}")
    print(f"  Info: {info}")
    print(f"  Done: {done}")
    print()
    
    if done:
        break

print(f"Total Reward: {total_reward:.2f}")

# Visualisasi posisi akhir
fig, ax = env.render()
plt.title('After Manual Steps')
plt.show()

## Bagian 2: Q-Learning Agent

Sekarang kita akan melatih agent menggunakan Q-Learning.

In [ ]:
# Inisialisasi environment dan agent
env = GridWorldSPG(grid_size=5, n_outlets=3, n_obstacles=3, seed=SEED)

n_states = env.grid_size ** 2
n_actions = len(env.actions)

q_agent = QLearningAgent(
    n_states=n_states,
    n_actions=n_actions,
    learning_rate=0.1,
    discount_factor=0.95,
    epsilon=1.0,
    epsilon_decay=0.995,
    epsilon_min=0.01,
    seed=SEED
)

print("Q-Learning Agent initialized!")
print(f"Q-Table shape: {q_agent.q_table.shape}")
print(f"Initial Q-values (all zeros): \n{q_agent.q_table[:3]}")

### 🎯 Latihan 2.1: Training Q-Learning Agent

In [ ]:
# Train agent
print("Training Q-Learning Agent...")
history = q_agent.train(env, n_episodes=1000, max_steps_per_episode=100)
print("\n✓ Training completed!")

# Cek Q-table setelah training
print(f"\nLearned Q-values (first 3 states):\n{q_agent.q_table[:3]}")

### 🎯 Latihan 2.2: Analisis Learning Curves

In [ ]:
# Plot learning curves
fig = plot_learning_curve(history, figsize=(15, 5))
plt.suptitle('Q-Learning: Learning Curves', fontsize=16, fontweight='bold', y=1.02)
plt.show()

# Analisis
print("\n📊 Training Analysis:")
print(f"First 10 episodes avg reward: {np.mean(history['rewards'][:10]):.2f}")
print(f"Last 10 episodes avg reward: {np.mean(history['rewards'][-10:]):.2f}")
print(f"\nFirst 10 episodes avg steps: {np.mean(history['steps'][:10]):.2f}")
print(f"Last 10 episodes avg steps: {np.mean(history['steps'][-10:]):.2f}")
print(f"\nFinal epsilon: {q_agent.epsilon:.4f}")

### 🎯 Latihan 2.3: Visualisasi Learned Policy

In [ ]:
# Visualisasi learned policy (best action per state)
env.reset()
fig, ax = env.render(q_table=q_agent.q_table)
plt.title('Learned Policy (arrows show best action per state)', fontsize=14, fontweight='bold')
plt.show()

print("\nInterpretasi:")
print("- Panah biru menunjukkan tindakan terbaik untuk setiap state")
print("- Agent telah belajar cara menghindari obstacle (X)")
print("- Agent mengarah ke outlet (O) terdekat")

## Bagian 3: Evaluasi dan Perbandingan

Bandingkan Q-Learning agent dengan random agent (baseline).

In [ ]:
# Evaluasi Q-Learning agent
print("Evaluating Q-Learning Agent...")
q_results = q_agent.evaluate(env, n_episodes=100, max_steps_per_episode=100)

# Evaluasi Random agent
print("Evaluating Random Agent...")
random_agent = RandomAgent(n_actions=n_actions, seed=SEED)
random_results = random_agent.evaluate(env, n_episodes=100, max_steps_per_episode=100)

# Perbandingan
compare_policies(q_results, random_results)

### 🎯 Latihan 3.1: Visualisasi Perbandingan

In [ ]:
# Plot perbandingan
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Reward comparison
agents = ['Q-Learning', 'Random']
mean_rewards = [q_results['mean_reward'], random_results['mean_reward']]
std_rewards = [q_results['std_reward'], random_results['std_reward']]

axes[0].bar(agents, mean_rewards, yerr=std_rewards, capsize=5, alpha=0.7, color=['green', 'red'])
axes[0].set_ylabel('Mean Reward')
axes[0].set_title('Reward Comparison')
axes[0].grid(True, alpha=0.3)

# Steps comparison
mean_steps = [q_results['mean_steps'], random_results['mean_steps']]
std_steps = [q_results['std_steps'], random_results['std_steps']]

axes[1].bar(agents, mean_steps, yerr=std_steps, capsize=5, alpha=0.7, color=['green', 'red'])
axes[1].set_ylabel('Mean Steps')
axes[1].set_title('Steps Comparison (lower is better)')
axes[1].grid(True, alpha=0.3)

# Success rate comparison
success_rates = [q_results['success_rate'] * 100, random_results['success_rate'] * 100]

axes[2].bar(agents, success_rates, alpha=0.7, color=['green', 'red'])
axes[2].set_ylabel('Success Rate (%)')
axes[2].set_title('Success Rate Comparison')
axes[2].set_ylim(0, 105)
axes[2].grid(True, alpha=0.3)

plt.suptitle('Q-Learning vs Random Agent', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## Bagian 4: Eksperimen dengan Hyperparameters

Coba variasi hyperparameters dan lihat pengaruhnya.

### 🎯 Latihan 4.1: Pengaruh Learning Rate

In [ ]:
# Coba berbagai learning rates
learning_rates = [0.01, 0.1, 0.3, 0.5]
results_by_lr = {}

for lr in learning_rates:
    print(f"\nTraining with learning_rate={lr}...")
    env = GridWorldSPG(grid_size=5, n_outlets=3, n_obstacles=3, seed=SEED)
    agent = QLearningAgent(
        n_states=n_states,
        n_actions=n_actions,
        learning_rate=lr,
        discount_factor=0.95,
        epsilon=1.0,
        epsilon_decay=0.995,
        epsilon_min=0.01,
        seed=SEED
    )
    history = agent.train(env, n_episodes=500, max_steps_per_episode=100)
    results_by_lr[lr] = history

print("\n✓ All experiments completed!")

In [ ]:
# Plot hasil
fig, ax = plt.subplots(figsize=(12, 6))

window = 50
for lr, history in results_by_lr.items():
    # Moving average
    if len(history['rewards']) >= window:
        moving_avg = np.convolve(history['rewards'], 
                                np.ones(window)/window, mode='valid')
        ax.plot(history['episodes'][window-1:], moving_avg, 
               linewidth=2, label=f'α={lr}')

ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Reward (50-episode moving avg)', fontsize=12)
ax.set_title('Effect of Learning Rate on Training', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Observasi:")
print("- Learning rate kecil (0.01): belajar lambat tapi stabil")
print("- Learning rate besar (0.5): belajar cepat tapi kurang stabil")
print("- Learning rate sedang (0.1-0.3): balance yang baik")

### 🎯 Latihan 4.2: Pengaruh Discount Factor

In [ ]:
# Coba berbagai discount factors
discount_factors = [0.5, 0.9, 0.95, 0.99]
results_by_gamma = {}

for gamma in discount_factors:
    print(f"\nTraining with discount_factor={gamma}...")
    env = GridWorldSPG(grid_size=5, n_outlets=3, n_obstacles=3, seed=SEED)
    agent = QLearningAgent(
        n_states=n_states,
        n_actions=n_actions,
        learning_rate=0.1,
        discount_factor=gamma,
        epsilon=1.0,
        epsilon_decay=0.995,
        epsilon_min=0.01,
        seed=SEED
    )
    history = agent.train(env, n_episodes=500, max_steps_per_episode=100)
    results_by_gamma[gamma] = history

print("\n✓ All experiments completed!")

In [ ]:
# Plot hasil
fig, ax = plt.subplots(figsize=(12, 6))

window = 50
for gamma, history in results_by_gamma.items():
    if len(history['rewards']) >= window:
        moving_avg = np.convolve(history['rewards'], 
                                np.ones(window)/window, mode='valid')
        ax.plot(history['episodes'][window-1:], moving_avg, 
               linewidth=2, label=f'γ={gamma}')

ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Reward (50-episode moving avg)', fontsize=12)
ax.set_title('Effect of Discount Factor on Training', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Observasi:")
print("- γ kecil (0.5): fokus short-term, mungkin kurang optimal")
print("- γ besar (0.99): pertimbangkan long-term rewards")
print("- γ=0.95: sering menjadi pilihan yang baik (balance)")

## Bagian 5: Challenge - Buat Environment Sendiri

Coba modifikasi environment untuk membuat task yang lebih challenging!

In [ ]:
# TODO: Buat environment dengan:
# - Grid size lebih besar (7x7 atau 10x10)
# - Lebih banyak outlets (5-7)
# - Lebih banyak obstacles (5-10)

# Your code here:
challenge_env = GridWorldSPG(
    grid_size=7,  # Ubah ini
    n_outlets=5,  # Ubah ini
    n_obstacles=7,  # Ubah ini
    seed=SEED
)

# Visualisasi
challenge_env.reset()
fig, ax = challenge_env.render()
plt.title('Challenge Environment', fontsize=14, fontweight='bold')
plt.show()

# Train agent pada challenge environment
print("\nTraining agent on challenge environment...")
n_states_challenge = challenge_env.grid_size ** 2
n_actions_challenge = len(challenge_env.actions)

challenge_agent = QLearningAgent(
    n_states=n_states_challenge,
    n_actions=n_actions_challenge,
    learning_rate=0.1,
    discount_factor=0.95,
    epsilon=1.0,
    epsilon_decay=0.995,
    epsilon_min=0.01,
    seed=SEED
)

challenge_history = challenge_agent.train(
    challenge_env, 
    n_episodes=2000,  # Lebih banyak episodes untuk task lebih sulit
    max_steps_per_episode=200
)

print("✓ Challenge training completed!")

In [ ]:
# Plot learning curves
fig = plot_learning_curve(challenge_history, figsize=(15, 5))
plt.suptitle('Challenge Environment: Learning Curves', fontsize=16, fontweight='bold', y=1.02)
plt.show()

# Visualisasi learned policy
challenge_env.reset()
fig, ax = challenge_env.render(q_table=challenge_agent.q_table)
plt.title('Challenge Environment: Learned Policy', fontsize=14, fontweight='bold')
plt.show()

# Evaluasi
challenge_results = challenge_agent.evaluate(
    challenge_env, 
    n_episodes=100, 
    max_steps_per_episode=200
)

print("\n📊 Challenge Environment Results:")
print(f"Mean Reward: {challenge_results['mean_reward']:.2f} ± {challenge_results['std_reward']:.2f}")
print(f"Mean Steps: {challenge_results['mean_steps']:.2f} ± {challenge_results['std_steps']:.2f}")
print(f"Success Rate: {challenge_results['success_rate']*100:.1f}%")

## 🎓 Kesimpulan

Dari latihan ini, kita telah belajar:

1. **Q-Learning Basics:**
   - Q-table menyimpan nilai action untuk setiap state
   - Update rule menggunakan reward dan max Q-value dari next state
   - Epsilon-greedy balance eksplorasi vs eksploitasi

2. **Training Process:**
   - Reward meningkat seiring training
   - Steps menurun (agent jadi lebih efisien)
   - Epsilon decay mengurangi eksplorasi bertahap

3. **Hyperparameter Effects:**
   - Learning rate: kecepatan vs stabilitas
   - Discount factor: short-term vs long-term
   - Epsilon decay: balance eksplorasi

4. **Evaluation:**
   - Q-Learning jauh lebih baik dari random
   - Success rate tinggi (>90%)
   - Lebih efisien (fewer steps)

**Next Steps:**
- Coba implement SARSA dan bandingkan dengan Q-Learning
- Explore continuous state spaces (Deep Q-Networks)
- Baca tentang Policy Gradient methods

**Selamat! Anda telah menguasai dasar-dasar Q-Learning! 🎉**